In [ ]:
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from evaluate import load
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"
#  You can install and import any other libraries if needed

In [ ]:
# Some Chinese punctuations will be tokenized as [UNK], so we replace them with English ones
token_replacement = [
    ["：" , ":"],
    ["，" , ","],
    ["“" , "\""],
    ["”" , "\""],
    ["？" , "?"],
    ["……" , "..."],
    ["！" , "!"]
]

In [ ]:

tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased", cache_dir="./cache/")

In [ ]:
class SemevalDataset(Dataset):
    def __init__(self, split="train") -> None:
        super().__init__()
        assert split in ["train", "validation", "test"]
        self.data = load_dataset(
            "sem_eval_2014_task_1", split=split, trust_remote_code=True, cache_dir="./cache/"
        ).to_list()

    def __getitem__(self, index):
        d = self.data[index]
        # Replace Chinese punctuations with English ones
        for k in ["premise", "hypothesis"]:
            for tok in token_replacement:
                d[k] = d[k].replace(tok[0], tok[1])
        return d

    def __len__(self):
        return len(self.data)

data_sample = SemevalDataset(split="train").data[:3]
print(f"Dataset example: \n{data_sample[0]} \n{data_sample[1]} \n{data_sample[2]}")

In [ ]:
# Define the hyperparameters
# You can modify these values if needed
lr = 3e-5
epochs = 3
train_batch_size = 8
validation_batch_size = 8

In [ ]:
# TODO1: Create batched data for DataLoader
# `collate_fn` is a function that defines how the data batch should be packed.
# This function will be called in the DataLoader to pack the data batch.

def collate_fn(batch):
    # TODO1-1: Implement the collate_fn function
    # Write your code here
    # The input parameter is a data batch (tuple), and this function packs it into tensors.
    # Use tokenizer to pack tokenize and pack the data and its corresponding labels.
    # Return the data batch and labels for each sub-task.
    premises = [d['premise'] for d in batch]
    hypotheses = [d['hypothesis'] for d in batch]
    inputs = tokenizer(premises, hypotheses, padding=True, truncation=True, return_tensors="pt")

    entailment_labels = [d['entailment'] for d in batch]
    relatedness_scores = [d['relatedness_score'] for d in batch]

    label_map = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
    entailment_labels_ids = torch.tensor([label_map[label] for label in entailment_labels], dtype=torch.long)
    relatedness_scores_tensor = torch.tensor(relatedness_scores, dtype=torch.float)
    
    batch = {}
    batch["input_ids"] = inputs.input_ids
    batch["attention_mask"] = inputs.attention_mask
    batch["entailment"] = entailment_labels_ids
    batch["relatedness"] = relatedness_scores_tensor
    return batch

# TODO1-2: Define your DataLoader
dl_train = DataLoader(SemevalDataset("train"), batch_size=train_batch_size, collate_fn=collate_fn)
dl_validation = DataLoader(SemevalDataset("validation"), batch_size=validation_batch_size, collate_fn=collate_fn)
dl_test = DataLoader(SemevalDataset("test"), batch_size=validation_batch_size, collate_fn=collate_fn)

In [ ]:
# TODO2: Construct your model
class MultiLabelModel(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Write your code here
        # Define what modules you will use in the model
        # Please use "google-bert/bert-base-uncased" model (https://huggingface.co/google-bert/bert-base-uncased)
        # Besides the base model, you may design additional architectures by incorporating linear layers, activation functions, or other neural components.
        # Remark: The use of any additional pretrained language models is not permitted.
        self.bert = BertModel.from_pretrained("google-bert/bert-base-uncased")
        self.dropout = torch.nn.Dropout(0.1)
        
        self.entailment_classifier = torch.nn.Linear(768, 3)
        
        self.relatedness_regressor = torch.nn.Linear(768, 1)
        
    def forward(self, **kwargs):
        # Write your code here
        # Forward pass
        input_ids = kwargs.get("input_ids")
        attention_mask = kwargs.get("attention_mask")
        
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        pooled_output = outputs.pooler_output  # Shape: (batch_size, 768)
        pooled_output = self.dropout(pooled_output)
        
        entailment_logits = self.entailment_classifier(pooled_output)  # Shape: (batch_size, 3)
        relatedness_score = self.relatedness_regressor(pooled_output).squeeze(-1)  # Shape: (batch_size,)
        
        return {
            "entailment_logits": entailment_logits,
            "relatedness_score": relatedness_score
        }

In [ ]:
# TODO3: Define your optimizer and loss function

model = MultiLabelModel().to(device)
# TODO3-1: Define your Optimizer
optimizer = AdamW(model.parameters(), lr=lr)

# TODO3-2: Define your loss functions (you should have two)
# Write your code here
criterion_entailment = torch.nn.CrossEntropyLoss()
criterion_relatedness = torch.nn.MSELoss()
# scoring functions
psr = load("pearsonr")
acc = load("accuracy")

In [ ]:
best_score = 0.0
for ep in range(epochs):
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train()
    # TODO4: Write the training loop
    # Write your code here
    # train your model
    # clear gradient
    # forward pass
    # compute loss
    # back-propagation
    # model optimization
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        
        loss_ent = criterion_entailment(outputs["entailment_logits"], batch["entailment"])
        loss_rel = criterion_relatedness(outputs["relatedness_score"], batch["relatedness"])
        
        total_loss = loss_ent + loss_rel
        
        total_loss.backward()
        optimizer.step()
        
        pbar.set_postfix({"loss_ent": loss_ent.item(), "loss_rel": loss_rel.item()})

    pbar = tqdm(dl_validation)
    pbar.set_description(f"Validation epoch [{ep+1}/{epochs}]")
    model.eval()
    # TODO5: Write the evaluation loop
    
    # Lists to collect all predictions and labels
    all_entailment_preds = []
    all_entailment_labels = []
    all_relatedness_preds = []
    all_relatedness_labels = []
    
    with torch.no_grad(): 
        for batch in pbar:

            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            
            entailment_preds = torch.argmax(outputs["entailment_logits"], dim=1)
            
            relatedness_preds = outputs["relatedness_score"]
            
            all_entailment_preds.extend(entailment_preds.cpu().tolist())
            all_entailment_labels.extend(batch["entailment"].cpu().tolist())
            all_relatedness_preds.extend(relatedness_preds.cpu().tolist())
            all_relatedness_labels.extend(batch["relatedness"].cpu().tolist())
    
    # Compute evaluation metrics
    pearson_corr = psr.compute(predictions=all_relatedness_preds, references=all_relatedness_labels)['pearsonr']
    accuracy = acc.compute(predictions=all_entailment_preds, references=all_entailment_labels)['accuracy']
    
    # Print results
    print(f"Epoch {ep+1}/{epochs} - Pearson Correlation: {pearson_corr:.4f}, Accuracy: {accuracy:.4f}")
    
    if pearson_corr + accuracy > best_score:
        best_score = pearson_corr + accuracy
        torch.save(model.state_dict(), f'./saved_models/best_model.ckpt')

In [ ]:
# Load the model
model = MultiLabelModel().to(device)
model.load_state_dict(torch.load(f"./saved_models/best_model.ckpt", weights_only=True))

# Test Loop
pbar = tqdm(dl_test, desc="Test")
model.eval()

# TODO6: Write the test loop

# Lists to collect all predictions and labels
all_entailment_preds = []
all_entailment_labels = []
all_relatedness_preds = []
all_relatedness_labels = []

with torch.no_grad():  # Disable gradient calculation for testing
    for batch in pbar:
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        
        entailment_preds = torch.argmax(outputs["entailment_logits"], dim=1)
        
        relatedness_preds = ou
        tputs["relatedness_score"]
        
        all_entailment_preds.extend(entailment_preds.cpu().tolist())
        all_entailment_labels.extend(batch["entailment"].cpu().tolist())
        all_relatedness_preds.extend(relatedness_preds.cpu().tolist())
        all_relatedness_labels.extend(batch["relatedness"].cpu().tolist())

test_pearson_corr = psr.compute(predictions=all_relatedness_preds, references=all_relatedness_labels)['pearsonr']
test_accuracy = acc.compute(predictions=all_entailment_preds, references=all_entailment_labels)['accuracy']


print(f"{'='*50}")
print(f"Test Pearson Correlation: {test_pearson_corr:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"{'='*50}")